# Serve Qwen3.8-27B on Kaggle free tier

Turns a free Kaggle GPU session into an **OpenAI-compatible endpoint** on a public HTTPS URL,
so the agent running on your laptop can call it.

## Sidebar setup, once

1. **Accelerator** = `GPU T4 x2`
2. **Internet** = `On`
3. **Add-ons -> Secrets** = add `HF_TOKEN`, and make sure its toggle is **attached to this notebook**

## How to use it

| Run | `BUILD_CACHE` | What you do |
|---|---|---|
| First | `True` | **Run All**, wait, then **Save Version -> Save & Run All**, then **Stop Session** |
| Every time after | `False` | **Add Input -> Notebook Output ->** this notebook, set `CACHE`, **Run All** |

The first run downloads 16.5 GB. Later runs skip that entirely.

## Stopping

**Stop Session** in the sidebar. Nothing else stops the quota meter. See the table at the bottom.

## Why Ollama, and not vLLM or llama-cpp-python

This was checked against release dates, not assumed.

**vLLM is wrong for this hardware.** vLLM 0.28 wants torch 2.13 and the CUDA 13 stack, so on Kaggle it
uninstalls and replaces torch, torchvision, triton, cudnn, cublas and roughly thirty other packages.
Worse, Turing (the T4, compute capability 7.5) is a neglected path in vLLM, with open reports of
*exactly* this configuration, a 27B Qwen on a T4, hanging indefinitely.

**llama-cpp-python cannot load this model.** The prebuilt CUDA wheel index tops out at **0.3.19,
released March 2026**. Qwen3.8-27B was released **13-14 August 2026** and registers a new `qwen35`
architecture; builds older than that **refuse the file outright**. Upstream llama-cpp-python is at
0.3.35 (17 Aug 2026) and would work, but has no prebuilt CUDA wheel, meaning a ~30 minute CMake
compile on every cache rebuild.

**Ollama fits.** Qwen3.8-27B support landed in **v0.32.12**, with follow-up fixes in v0.32.13 and
v0.32.14 for Qwen3.8 system-message handling. We pin v0.33.2 (27 Aug 2026), comfortably past all of it.

| | vLLM | llama-cpp-python | **Ollama** |
|---|---|---|---|
| Supports Qwen3.8 arch | Yes | **No** (wheel is pre-release) | **Yes**, v0.32.12+ |
| Turing / T4 | Neglected, documented hangs | Good | **Supported, cc 5.0+** |
| Touches Kaggle's CUDA stack | Replaces it | Links against it | **Ignores it, bundles its own** |
| Install | ~4 GB of wheels | One wheel + compile risk | **One 1.3 GB tarball** |
| Model size | 19.5 GB AWQ | 16.5 GB GGUF | **16.5 GB GGUF** |

The third row is the one that matters most. Ollama ships its own CUDA runtime, so whatever CUDA
version Kaggle's image happens to be on this month is **irrelevant**. Only the NVIDIA driver
matters, and that is fixed by the host.

### Two footguns this notebook works around

- `https://ollama.com/download/ollama-linux-amd64.tgz` **404s as of v0.33.2.** The asset was renamed
  to `ollama-linux-amd64.tar.zst`. Most tutorials online are stale.
- **Ollama has no authentication.** A Cloudflare tunnel would put an open 27B endpoint on the public
  internet. A small bearer-token proxy sits in front of it.

In [ ]:
# ============================================================
#  CONFIG - the only cell you edit
# ============================================================

BUILD_CACHE = False    # True for the one-time cache build. False every run after.

CACHE = "/kaggle/input/qwen27b-ollama-cache"   # path shown by Add Input

RUN_MINUTES = 120      # auto-stop the model after this long
N_CTX       = 8192     # context window. Lower to 4096 if memory is tight.

OLLAMA_VER  = "v0.33.2"    # pinned. Qwen3.8 needs >= v0.32.12
MODEL_NAME  = "qwen"       # the name your client passes as "model"

REPO = "unsloth/Qwen3.8-27B-GGUF"
FILE = "Qwen3.8-27B-UD-Q4_K_M.gguf"    # 16.5 GB. Quality/size sweet spot.
# smaller: Qwen3.8-27B-UD-IQ4_XS.gguf   14.3 GB
# bigger:  Qwen3.8-27B-UD-Q5_K_M.gguf   19.8 GB, too big for the 20 GB cache
# We deliberately skip mmproj-F16.gguf: that is the vision encoder, and this
# project only ever sends text.

import os, sys, time, datetime, subprocess, shutil, json
T0 = time.time()

def step(msg):
    # Timestamped progress, so a long pause is visibly not a hang.
    print(f"[{datetime.datetime.now():%H:%M:%S}  +{time.time()-T0:5.0f}s]  {msg}", flush=True)

def sh(cmd, **kw):
    # Run a command, stream its output, raise on failure.
    return subprocess.run(cmd, check=True, **kw)

# --- HF token from Kaggle Secrets -------------------------------------------
# Never paste a token into a cell. Add-ons -> Secrets -> HF_TOKEN, then attach
# it to this notebook with the toggle.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    step(f"HF_TOKEN loaded from Kaggle Secrets ({len(HF_TOKEN)} chars, not printed)")
except Exception as e:
    step(f"HF_TOKEN not available: {type(e).__name__}: {e}")
    step("  unsloth/Qwen3.8-27B-GGUF is public, so the download should still work.")
    step("  If it 401s: Add-ons -> Secrets -> add HF_TOKEN -> toggle it on for this notebook.")

# --- paths -------------------------------------------------------------------
# Ollama itself goes to /kaggle/tmp (~60 GB, not persisted, re-downloaded each run).
# Only the model store goes to /kaggle/working, which is capped at 20 GB.
OLLAMA_HOME = "/kaggle/tmp/ollama"
STORE = "/kaggle/working/ollama-models" if BUILD_CACHE else "/kaggle/tmp/ollama-models"

CACHE_DONE = False
HAVE_CACHE = (not BUILD_CACHE) and os.path.isdir(f"{CACHE}/ollama-models/manifests")

step("MODE: " + ("BUILD CACHE (one-time)" if BUILD_CACHE
                 else ("serve from cache" if HAVE_CACHE else "serve, NO CACHE - will download 16.5 GB")))
if not BUILD_CACHE and not HAVE_CACHE:
    step(f"  nothing found at {CACHE}/ollama-models/")
    step("  Add Input -> Notebook Output -> this notebook, and check CACHE above.")

In [ ]:
# ============================================================
#  1. Environment probe. Confirms the assumptions above hold.
# ============================================================

step("GPU")
gpu = subprocess.run(["nvidia-smi",
    "--query-gpu=index,name,memory.total,compute_cap,driver_version",
    "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)

n_gpu = len([l for l in gpu.splitlines() if l.strip()])
assert n_gpu >= 1, "No GPU. Sidebar -> Accelerator -> GPU T4 x2, then Run All again."
if n_gpu < 2:
    print(f"\n  WARNING: {n_gpu} GPU visible. A 16.5 GB model does not fit one 15.6 GB T4.")
    print("  Sidebar -> Accelerator -> 'GPU T4 x2'.")

step("driver / CUDA")
# 'CUDA Version' here is the MAXIMUM the driver supports, not what is installed.
# It is the only CUDA number that matters to us: Ollama bundles its own runtime.
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout.splitlines()[2])

step("python / disk")
print("  python  :", sys.version.split()[0])
print("  ", subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/tmp"],
                           capture_output=True, text=True).stdout.replace("\n", "\n   "))
print("  zstd    :", shutil.which("zstd") or "not on PATH (python fallback will be used)")

In [ ]:
# ============================================================
#  2. Install Ollama and start the server.
# ============================================================

# NOTE: https://ollama.com/download/ollama-linux-amd64.tgz returns 404 as of
# v0.33.2 - the asset was renamed to .tar.zst. Most guides online are stale.
URL = (f"https://github.com/ollama/ollama/releases/download/{OLLAMA_VER}"
       f"/ollama-linux-amd64.tar.zst")
TARBALL = "/kaggle/tmp/ollama.tar.zst"

os.makedirs("/kaggle/tmp", exist_ok=True)
os.makedirs(OLLAMA_HOME, exist_ok=True)

if not os.path.exists(f"{OLLAMA_HOME}/bin/ollama"):
    step(f"downloading Ollama {OLLAMA_VER} (~1.3 GB)")
    sh(["wget", "--progress=bar:force:noscroll", "-O", TARBALL, URL])

    step("extracting")
    try:
        sh(["tar", "--use-compress-program=unzstd", "-xf", TARBALL, "-C", OLLAMA_HOME])
    except Exception as e:
        step(f"  system zstd failed ({e}); falling back to python zstandard")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zstandard"], check=True)
        import zstandard, tarfile
        with open(TARBALL, "rb") as f, zstandard.ZstdDecompressor().stream_reader(f) as r:
            with tarfile.open(fileobj=r, mode="r|") as t:
                t.extractall(OLLAMA_HOME)
    os.remove(TARBALL)

OLLAMA = f"{OLLAMA_HOME}/bin/ollama"
os.chmod(OLLAMA, 0o755)
step(f"installed: {subprocess.run([OLLAMA, '--version'], capture_output=True, text=True).stdout.strip()}")

# --- wire the cached model store --------------------------------------------
# /kaggle/input is read-only and Ollama needs to write. Blobs are content
# addressed and immutable, so we symlink those and copy only the tiny manifests.
os.makedirs(f"{STORE}/blobs", exist_ok=True)
if HAVE_CACHE:
    step("linking cached model store (symlinked blobs, no 16.5 GB copy)")
    src = f"{CACHE}/ollama-models"
    if os.path.isdir(f"{STORE}/manifests"):
        shutil.rmtree(f"{STORE}/manifests")
    shutil.copytree(f"{src}/manifests", f"{STORE}/manifests")
    linked = 0
    for b in os.listdir(f"{src}/blobs"):
        dst = f"{STORE}/blobs/{b}"
        if not os.path.exists(dst):
            os.symlink(f"{src}/blobs/{b}", dst); linked += 1
    step(f"  {linked} blobs linked")

# --- start the server --------------------------------------------------------
env = {**os.environ,
       "OLLAMA_HOME": OLLAMA_HOME,
       "OLLAMA_MODELS": STORE,
       "OLLAMA_HOST": "127.0.0.1:11434",
       "OLLAMA_KEEP_ALIVE": "24h",        # default is 5m; an unload means a slow reload
       "OLLAMA_CONTEXT_LENGTH": str(N_CTX),
       "OLLAMA_FLASH_ATTENTION": "1"}

SRV_LOG = "/kaggle/working/ollama-serve.log"
serve = subprocess.Popen([OLLAMA, "serve"], env=env,
                         stdout=open(SRV_LOG, "w"), stderr=subprocess.STDOUT)

import urllib.request, urllib.error
for i in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=3)
        step(f"ollama serve up, pid {serve.pid}"); break
    except Exception:
        time.sleep(2)
else:
    print("".join(open(SRV_LOG).readlines()[-40:]))
    raise RuntimeError("ollama serve did not come up")

# Confirm Ollama found the GPUs. If this says CPU, everything below will crawl.
print("  " + "\n  ".join(l.rstrip()[:160] for l in open(SRV_LOG)
                         if any(k in l for k in ("inference compute", "library=cuda",
                                                 "gpu", "GPU", "total blobs")))[:2000])

In [ ]:
# ============================================================
#  3. Get the model into the store.
# ============================================================

MF = "/kaggle/tmp/Modelfile"

def write_modelfile(gguf_path):
    # Sampling defaults are the ones Qwen publishes for non-thinking mode.
    # Any API request can override them per call.
    # No TEMPLATE line: Ollama reads tokenizer.chat_template from GGUF metadata.
    lines = [f"FROM {gguf_path}",
             f"PARAMETER num_ctx {N_CTX}",
             "PARAMETER num_gpu 999",
             "PARAMETER temperature 0.7",
             "PARAMETER top_p 0.8",
             "PARAMETER top_k 20",
             "PARAMETER min_p 0.0"]
    open(MF, "w").write(chr(10).join(lines) + chr(10))

if BUILD_CACHE:
    step(f"downloading {FILE} (~16.5 GB, the slow part)")
    from huggingface_hub import hf_hub_download
    # hf_hub_download, NOT snapshot_download: the repo holds 27 quants and we want one.
    gguf = hf_hub_download(repo_id=REPO, filename=FILE,
                           local_dir="/kaggle/tmp/gguf", token=HF_TOKEN)
    step(f"  got {os.path.getsize(gguf)/2**30:.2f} GiB")

    step(f"importing into Ollama as '{MODEL_NAME}' (copies the blob, a few minutes)")
    write_modelfile(gguf)
    sh([OLLAMA, "create", MODEL_NAME, "-f", MF], env=env)
    os.remove(gguf)   # the blob now lives in the store; do not pay for it twice

    size = int(subprocess.run(["du", "-sb", "/kaggle/working"],
                              capture_output=True, text=True).stdout.split()[0])
    step(f"/kaggle/working = {size/2**30:.2f} GiB")
    if size > 19 * 2**30:
        print("  WARNING: close to Kaggle's 20 GB output cap. Save Version may fail.")
        print("  Switch FILE to Qwen3.8-27B-UD-IQ4_XS.gguf (14.3 GB) and rebuild.")

    CACHE_DONE = True
    print("\n" + "=" * 70)
    print("  CACHE BUILT. Next:")
    print("    1. Save Version -> Save & Run All   (publishes /kaggle/working)")
    print("    2. wait for it to finish, then Stop Session")
    print("    3. set BUILD_CACHE = False")
    print("    4. Add Input -> Notebook Output -> this notebook; set CACHE")
    print("=" * 70)

elif not HAVE_CACHE:
    step("no cache attached: pulling from Ollama's registry instead (~18 GB)")
    sh([OLLAMA, "pull", "qwen3.8:27b"], env=env)
    MODEL_NAME = "qwen3.8:27b"

step("models available:")
print(subprocess.run([OLLAMA, "list"], env=env, capture_output=True, text=True).stdout)

In [ ]:
# ============================================================
#  4. Auth proxy. Ollama has NO authentication of its own, and the
#     next cell puts it on the public internet.
# ============================================================

# SystemExit does not stop 'Run All' in Jupyter, so the guard is explicit.
if CACHE_DONE:
    raise RuntimeError("Cache build finished. Save Version, set BUILD_CACHE=False, "
                       "attach the output, then rerun.")

import secrets, threading, http.server, socketserver, urllib.request, urllib.error

PORT, API_KEY = 8000, secrets.token_urlsafe(24)
UPSTREAM = "http://127.0.0.1:11434"
HOP = {"connection", "keep-alive", "transfer-encoding", "content-length", "host"}

class Proxy(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, fmt, *a):
        # BaseHTTPRequestHandler calls this from log_request AND log_error with
        # different arities, so index nothing.
        print(f"    proxy {self.command} {self.path}", flush=True)

    def _authed(self):
        if self.headers.get("Authorization", "") == f"Bearer {API_KEY}":
            return True
        self.send_response(401)
        self.send_header("Content-Length", "0")
        self.end_headers()
        return False

    def _forward(self, body=None):
        if not self._authed():
            return
        req = urllib.request.Request(UPSTREAM + self.path, data=body, method=self.command)
        for k, v in self.headers.items():
            if k.lower() not in HOP and k.lower() != "authorization":
                req.add_header(k, v)
        try:
            up = urllib.request.urlopen(req, timeout=900)
        except urllib.error.HTTPError as e:
            up = e
        except Exception as e:
            self.send_response(502); self.send_header("Content-Length", "0"); self.end_headers()
            print(f"    proxy upstream error: {e}", flush=True)
            return
        self.send_response(up.status)
        for k, v in up.headers.items():
            if k.lower() not in HOP:
                self.send_header(k, v)
        # Chunked, so streamed completions arrive token by token instead of
        # being buffered until the request completes.
        self.send_header("Transfer-Encoding", "chunked")
        self.end_headers()
        try:
            while True:
                # readline, not read(n): read(n) blocks until n bytes arrive,
                # which would defeat streaming. SSE is line delimited.
                chunk = up.readline()
                if not chunk:
                    break
                self.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                self.wfile.flush()
            self.wfile.write(b"0\r\n\r\n")
            self.wfile.flush()
        except (BrokenPipeError, ConnectionResetError):
            pass

    def do_GET(self):
        self._forward()

    def do_POST(self):
        n = int(self.headers.get("Content-Length") or 0)
        self._forward(self.rfile.read(n) if n else b"")

class Threaded(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True
    allow_reuse_address = True

proxy = Threaded(("127.0.0.1", PORT), Proxy)
threading.Thread(target=proxy.serve_forever, daemon=True).start()
step(f"auth proxy listening on {PORT}, forwarding to {UPSTREAM}")

In [ ]:
# ============================================================
#  5. Public tunnel, then an end-to-end test through it.
# ============================================================

if CACHE_DONE:
    raise RuntimeError("cache build run - stop here")

import re, requests

if not shutil.which("cloudflared"):
    step("installing cloudflared")
    sh(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

# Load the model over localhost FIRST. Cloudflare's edge will drop a request
# that sits silent for minutes, and the first call has to read 16.5 GB onto the
# GPUs. Warming locally means the tunnel's first request is already fast.
step("warming the model on localhost - reads 16.5 GB onto the GPUs, several minutes")
w0 = time.time()
warm = requests.post("http://127.0.0.1:11434/v1/chat/completions",
                     json={"model": MODEL_NAME, "max_tokens": 8, "temperature": 0,
                           "messages": [{"role": "user", "content": "hi"}]},
                     timeout=2400)
if not warm.ok:
    print(warm.text[:800]); print("".join(open(SRV_LOG).readlines()[-30:]))
    raise RuntimeError(f"model failed to load ({warm.status_code})")
step(f"model loaded and generating in {(time.time()-w0)/60:.1f} min")

# Where did the layers actually land? If this shows CPU, throughput will be unusable.
print(subprocess.run([OLLAMA, "ps"], env=env, capture_output=True, text=True).stdout)

TUN_LOG = "/kaggle/working/tunnel.log"
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open(TUN_LOG, "w"), stderr=subprocess.STDOUT)

PUBLIC = None
for _ in range(40):
    time.sleep(3)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TUN_LOG).read())
    if m:
        PUBLIC = m.group(0); break
assert PUBLIC, f"tunnel failed, see {TUN_LOG}"
step(f"tunnel up: {PUBLIC}")

step("end-to-end test through the tunnel (model is already warm)")
r = requests.post(PUBLIC + "/v1/chat/completions",
                  headers={"Authorization": f"Bearer {API_KEY}"},
                  json={"model": MODEL_NAME, "max_tokens": 16, "temperature": 0,
                        "messages": [{"role": "user", "content": "Reply with exactly: ready"}]},
                  timeout=300)

# The auth check must actually reject. Verify it rather than assume it.
unauth = requests.get(PUBLIC + "/v1/models", timeout=60).status_code

print("\n" + "=" * 70)
print("  ENDPOINT LIVE" if r.ok else f"  TUNNEL UP BUT TEST FAILED ({r.status_code})")
print("=" * 70)
print(f"  OPENAI_BASE_URL = {PUBLIC}/v1")
print(f"  OPENAI_API_KEY  = {API_KEY}")
print(f"  model           = {MODEL_NAME}")
print(f"  auth check      = {unauth} without a token " +
      ("(correctly rejected)" if unauth == 401 else "*** NOT PROTECTED ***"))
print(f"  setup took {(time.time()-T0)/60:.1f} min")
print("=" * 70)
if r.ok:
    print("  reply:", r.json()["choices"][0]["message"]["content"].strip()[:80])
else:
    print(r.text[:500])
    print("".join(open(SRV_LOG).readlines()[-25:]))
print("\n  URL and key are new every session. Put them in a .env, never in git.")

In [ ]:
# ============================================================
#  6. Hold the endpoint open, then stop the model automatically.
#     Interrupt this cell to stop sooner.
# ============================================================

if CACHE_DONE:
    raise RuntimeError("cache build run - stop here")

end = time.time() + RUN_MINUTES * 60
step(f"serving until {datetime.datetime.fromtimestamp(end):%H:%M:%S}")
try:
    while time.time() < end:
        if serve.poll() is not None:
            print("ollama serve died:")
            print("".join(open(SRV_LOG).readlines()[-30:]))
            break
        step(f"alive, {int((end-time.time())/60)} min left  |  {PUBLIC}")
        time.sleep(300)
finally:
    for p in (tunnel, serve):
        try: p.terminate()
        except Exception: pass
    try: proxy.shutdown()
    except Exception: pass
    time.sleep(5)
    print("\n" + "=" * 70)
    print("  Model stopped. GPU memory is free. Tunnel is closed.")
    print("")
    print("  YOUR QUOTA IS STILL RUNNING.")
    print("  Kaggle charges for an ATTACHED GPU session, not for GPU work.")
    print("  Click STOP SESSION in the sidebar. That is the only thing")
    print("  that stops the meter.")
    print("=" * 70)

## What actually stops the quota

| Action | Frees GPU memory | **Stops quota** |
|---|---|---|
| Timer expires, or you interrupt the cell | Yes | **No** |
| Closing the browser tab | No | **No** |
| **Stop Session** in the sidebar | Yes | **Yes** |
| Kaggle's idle timeout | Yes | Yes, eventually |

Kaggle bills for a GPU session being **attached**, not for the GPU doing work. The 30 hours per week
drain at the same rate whether the model is generating or sitting idle.

## Calling it from the laptop

```python
from openai import OpenAI
client = OpenAI(base_url="<OPENAI_BASE_URL>", api_key="<OPENAI_API_KEY>")
print(client.chat.completions.create(
    model="qwen",
    messages=[{"role": "user", "content": "hello"}],
).choices[0].message.content)
```

Both values change every session, because the Cloudflare quick tunnel gets a new random hostname and
the key is regenerated each run. Write them into a local `.env`, which `.gitignore` already excludes.

## If it fails

| Symptom | Cause and fix |
|---|---|
| `wget` 404 on the tarball | Ollama moved or yanked the pinned release. Bump `OLLAMA_VER` |
| Log says `library=cpu`, generation crawls | Ollama did not see the GPUs. Check the probe cell showed two |
| Out of memory on load | Drop `N_CTX` to 4096, or switch `FILE` to `UD-IQ4_XS` (14.3 GB) |
| `Error: unknown architecture` | Ollama is older than v0.32.12. Raise `OLLAMA_VER` |
| Only one GPU listed | Accelerator is not set to **T4 x2** |
| Save Version fails on the build run | `/kaggle/working` exceeded 20 GB. Use a smaller quant |
| 401 from your own client | The key is regenerated each session. Copy the new one |
| HF download 401s | Secret not attached. Add-ons -> Secrets -> toggle `HF_TOKEN` on |
| URL stops working | Session ended. Run All again for a new URL and key |

## Honest caveats

- **This has not been run on real hardware yet.** The version compatibility, asset URLs, file sizes
  and memory arithmetic were each verified against primary sources, but the first run is the test.
- **Cloudflare quick tunnels are not a production ingress.** They are rate limited, unauthenticated
  at the tunnel layer, and the hostname is public. The bearer proxy is what keeps the endpoint from
  being open to anyone who stumbles on the URL. Do not treat the URL itself as a secret.
- **No personal health data should ever cross this endpoint** while it is on a public tunnel. During
  development the agent sends synthetic or public-dataset records. That constraint is in the
  proposal, not an afterthought here.
- **Ollama is slower than vLLM under concurrency.** Irrelevant here: one user, occasional calls.
- **Symlinked blobs assume the cached dataset stays attached** for the whole session. It does.

In [ ]:
# OPTIONAL FALLBACK: end the container from inside the notebook.
# Kills every process, which terminates the session and stops the quota.
# Blunt: anything unsaved in /kaggle/working is lost. Prefer Stop Session.

CONFIRM = False   # set True, then run this cell

if CONFIRM:
    import signal
    print("terminating container...")
    os.kill(1, signal.SIGKILL)
else:
    print("Not armed. Set CONFIRM = True to terminate.")
    print("Prefer Stop Session in the sidebar.")